In [5]:
import os
import glob

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

print("All libraries imported successfully!")

Matplotlib is building the font cache; this may take a moment.


All libraries imported successfully!


-Exploring the Files and its Structure 

In [ ]:
#reading the names of the files in the directory
from pathlib import Path

from matplotlib.pylab import rint    
data_path = Path(r"C:\data2020")
files = sorted(data_path.glob("*.nc"))
print(f"Found {len(files)} NetCDF files.")

# Show the first 5 filenames
for file in files[:5]:
    print(file.name)




Found 88864 NetCDF files.
HGT-100mb_2020010100_ncum_imdaa_reanl_prl_00_100_hpa.nc
HGT-100mb_2020010103_ncum_imdaa_reanl_prl_03_100_hpa.nc
HGT-100mb_2020010106_ncum_imdaa_reanl_prl_06_100_hpa.nc
HGT-100mb_2020010109_ncum_imdaa_reanl_prl_09_100_hpa.nc
HGT-100mb_2020010112_ncum_imdaa_reanl_prl_12_100_hpa.nc


In [3]:
#structure of the file
import xarray as xr

ds = xr.open_dataset(files[0])
print(ds)

c:\Users\Kritika\Desktop\data_wind_prediction\.venv\Lib\site-packages\xarray\backends\plugins.py:109: RuntimeWarning: Engine 'cfgrib' loading failed:
Cannot find the ecCodes library
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


<xarray.Dataset> Size: 2MB
Dimensions:  (time: 1, plev: 1, lat: 501, lon: 751)
Coordinates:
  * time     (time) datetime64[ns] 8B 2020-01-01
  * plev     (plev) float64 8B 1e+04
  * lat      (lat) float64 4kB -15.0 -14.88 -14.76 -14.64 ... 44.76 44.88 45.0
  * lon      (lon) float64 6kB 30.0 30.12 30.24 30.36 ... 119.8 119.9 120.0
Data variables:
    gh       (time, plev, lat, lon) float32 2MB ...
Attributes:
    CDI:          Climate Data Interface version 2.4.0 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Fri Jun 12 22:47:29 2026: cdo -f nc4c -z zip_4 sellonlatbox...
    CDO:          Climate Data Operators version 2.4.0 (https://mpimet.mpg.de...


In [5]:
print(ds["gh"])
print(ds["gh"].values)

<xarray.DataArray 'gh' (time: 1, plev: 1, lat: 501, lon: 751)> Size: 2MB
[376251 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 8B 2020-01-01
  * plev     (plev) float64 8B 1e+04
  * lat      (lat) float64 4kB -15.0 -14.88 -14.76 -14.64 ... 44.76 44.88 45.0
  * lon      (lon) float64 6kB 30.0 30.12 30.24 30.36 ... 119.8 119.9 120.0
Attributes:
    standard_name:  geopotential_height
    long_name:      Geopotential height
    units:          gpm
    param:          5.3.0
[[[[nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   ...
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]
   [nan nan nan ... nan nan nan]]]]


In [6]:
import numpy as np

gh = ds["gh"].values

print("Shape:", gh.shape)
print("Minimum:", np.nanmin(gh))
print("Maximum:", np.nanmax(gh))
print("Mean:", np.nanmean(gh))

Shape: (1, 1, 501, 751)
Minimum: 15680.607
Maximum: 16997.107
Mean: 16421.51


In [ ]:
#converting data set into table format
df = ds.to_dataframe().reset_index()
print(df.head())

        time     plev   lat    lon  gh
0 2020-01-01  10000.0 -15.0  30.00 NaN
1 2020-01-01  10000.0 -15.0  30.12 NaN
2 2020-01-01  10000.0 -15.0  30.24 NaN
3 2020-01-01  10000.0 -15.0  30.36 NaN
4 2020-01-01  10000.0 -15.0  30.48 NaN


In [10]:
# checking the rows with NaN values in the 'gh' column
nan_rows = df[df['gh'].isna()]
print(f"Number of rows with NaN values in 'gh' column: {len(nan_rows)}")
# checking the rows without NaN values in the 'gh' column
non_nan_rows = df[df['gh'].notna()]
print(f"Number of rows without NaN values in 'gh' column: {len(non_nan_rows)}") 


Number of rows with NaN values in 'gh' column: 9531
Number of rows without NaN values in 'gh' column: 366720


    PROCESSING THE DATA SET OF YEAR 2020 (it is just for 1 time stamp in order to check what to do with all the files)
    - Dropping columns with "NaN" gh
    - Converting the Pressure unit from "Pa" to "hPa"

In [15]:
# dropping rows with NaN values in the 'gh' column
df = df.dropna(subset=['gh'])
print(f"Number of rows after dropping NaN values in 'gh' column: {len(df)}")
print(df.shape)
print(df.head())

Number of rows after dropping NaN values in 'gh' column: 366720
(366720, 5)
         time     plev   lat    lon            gh
33 2020-01-01  10000.0 -15.0  33.96  16744.607422
34 2020-01-01  10000.0 -15.0  34.08  16811.607422
35 2020-01-01  10000.0 -15.0  34.20  16776.107422
36 2020-01-01  10000.0 -15.0  34.32  16604.107422
37 2020-01-01  10000.0 -15.0  34.44  16501.607422


In [18]:
#checking the pressure units in the dataset
print(ds["plev"])

<xarray.DataArray 'plev' (plev: 1)> Size: 8B
array([10000.])
Coordinates:
  * plev     (plev) float64 8B 1e+04
Attributes:
    standard_name:  air_pressure
    long_name:      pressure
    units:          Pa
    positive:       down
    axis:           Z


In [ ]:
df["plev"] = df["plev"] / 100  # Convert pressure from Pa to hPa
print(df.head())


         time   plev   lat    lon            gh
33 2020-01-01  100.0 -15.0  33.96  16744.607422
34 2020-01-01  100.0 -15.0  34.08  16811.607422
35 2020-01-01  100.0 -15.0  34.20  16776.107422
36 2020-01-01  100.0 -15.0  34.32  16604.107422
37 2020-01-01  100.0 -15.0  34.44  16501.607422


In [42]:
df.rename(columns={"plev": "pressure_hPa"}, inplace=True)
print(df.head())

         time  pressure_hPa   lat    lon            gh
33 2020-01-01         100.0 -15.0  33.96  16744.607422
34 2020-01-01         100.0 -15.0  34.08  16811.607422
35 2020-01-01         100.0 -15.0  34.20  16776.107422
36 2020-01-01         100.0 -15.0  34.32  16604.107422
37 2020-01-01         100.0 -15.0  34.44  16501.607422


In [44]:
df = df.rename(columns={
    "pressure_hPa": "pressure_hpa",
    "lat": "latitude",
    "lon": "longitude",
    "gh": "geopotential_height"
})
print(df.columns)

Index(['time', 'pressure_hpa', 'latitude', 'longitude', 'geopotential_height'], dtype='str')


In [ ]:
df.info()

<class 'pandas.DataFrame'>
Index: 366720 entries, 33 to 376234
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   time                 366720 non-null  datetime64[ns]
 1   pressure_hpa         366720 non-null  float64       
 2   latitude             366720 non-null  float64       
 3   longitude            366720 non-null  float64       
 4   geopotential_height  366720 non-null  float32       
dtypes: datetime64[ns](1), float32(1), float64(3)
memory usage: 15.4 MB


In [47]:
print(df.head())

         time  pressure_hpa  latitude  longitude  geopotential_height
33 2020-01-01         100.0     -15.0      33.96         16744.607422
34 2020-01-01         100.0     -15.0      34.08         16811.607422
35 2020-01-01         100.0     -15.0      34.20         16776.107422
36 2020-01-01         100.0     -15.0      34.32         16604.107422
37 2020-01-01         100.0     -15.0      34.44         16501.607422


now for Tmp File of the same time stamp of hgt

In [48]:
tmp_file = next(f for f in files if "TMP-100mb_2020010100" in f.name)

print(tmp_file)

C:\data2020\TMP-100mb_2020010100_ncum_imdaa_reanl_prl_00_100_hpa.nc


In [49]:
tmp_ds = xr.open_dataset(tmp_file)

print(tmp_ds)

<xarray.Dataset> Size: 2MB
Dimensions:  (time: 1, plev: 1, lat: 501, lon: 751)
Coordinates:
  * time     (time) datetime64[ns] 8B 2020-01-01
  * plev     (plev) float64 8B 1e+04
  * lat      (lat) float64 4kB -15.0 -14.88 -14.76 -14.64 ... 44.76 44.88 45.0
  * lon      (lon) float64 6kB 30.0 30.12 30.24 30.36 ... 119.8 119.9 120.0
Data variables:
    t        (time, plev, lat, lon) float32 2MB ...
Attributes:
    CDI:          Climate Data Interface version 2.4.0 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Fri Jun 12 22:47:35 2026: cdo -f nc4c -z zip_4 sellonlatbox...
    CDO:          Climate Data Operators version 2.4.0 (https://mpimet.mpg.de...


In [50]:
print(tmp_ds["t"].attrs)

{'standard_name': 'air_temperature', 'long_name': 'Temperature', 'units': 'K', 'param': '0.0.0'}


In [51]:
tmp_df = tmp_ds.to_dataframe().reset_index()

tmp_df = tmp_df.dropna(subset=["t"])

tmp_df["plev"] = tmp_df["plev"] / 100

tmp_df = tmp_df.rename(columns={
    "plev": "pressure_hpa",
    "lat": "latitude",
    "lon": "longitude",
    "t": "temperature"
})

In [52]:
print(tmp_df.head())
print(tmp_df.info())

         time  pressure_hpa  latitude  longitude  temperature
33 2020-01-01         100.0     -15.0      33.96   190.821976
34 2020-01-01         100.0     -15.0      34.08   191.275101
35 2020-01-01         100.0     -15.0      34.20   190.993851
36 2020-01-01         100.0     -15.0      34.32   189.821976
37 2020-01-01         100.0     -15.0      34.44   189.243851
<class 'pandas.DataFrame'>
Index: 366720 entries, 33 to 376234
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   time          366720 non-null  datetime64[ns]
 1   pressure_hpa  366720 non-null  float64       
 2   latitude      366720 non-null  float64       
 3   longitude     366720 non-null  float64       
 4   temperature   366720 non-null  float32       
dtypes: datetime64[ns](1), float32(1), float64(3)
memory usage: 15.4 MB
None


In [53]:
#merging the two datasets on common columns
print(df.columns)
print(tmp_df.columns)

Index(['time', 'pressure_hpa', 'latitude', 'longitude', 'geopotential_height'], dtype='str')
Index(['time', 'pressure_hpa', 'latitude', 'longitude', 'temperature'], dtype='str')


In [54]:
merged_df = df.merge(
    tmp_df,
    on=["time", "pressure_hpa", "latitude", "longitude"],
    how="inner"
)

In [55]:
print("HGT shape:", df.shape)
print("TMP shape:", tmp_df.shape)
print("Merged shape:", merged_df.shape)

HGT shape: (366720, 5)
TMP shape: (366720, 5)
Merged shape: (366720, 6)


In [56]:
merged_df.head()

,time,pressure_hpa,latitude,longitude,geopotential_height,temperature
0,2020-01-01,100.0,-15.0,33.96,16744.607422,190.821976
1,2020-01-01,100.0,-15.0,34.08,16811.607422,191.275101
2,2020-01-01,100.0,-15.0,34.20,16776.107422,190.993851
3,2020-01-01,100.0,-15.0,34.32,16604.107422,189.821976
4,2020-01-01,100.0,-15.0,34.44,16501.607422,189.243851


In [57]:
print(merged_df.shape)
print(merged_df.info())
print(merged_df.head())

(366720, 6)
<class 'pandas.DataFrame'>
RangeIndex: 366720 entries, 0 to 366719
Data columns (total 6 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   time                 366720 non-null  datetime64[ns]
 1   pressure_hpa         366720 non-null  float64       
 2   latitude             366720 non-null  float64       
 3   longitude            366720 non-null  float64       
 4   geopotential_height  366720 non-null  float32       
 5   temperature          366720 non-null  float32       
dtypes: datetime64[ns](1), float32(2), float64(3)
memory usage: 14.0 MB
None
        time  pressure_hpa  latitude  longitude  geopotential_height  \
0 2020-01-01         100.0     -15.0      33.96         16744.607422   
1 2020-01-01         100.0     -15.0      34.08         16811.607422   
2 2020-01-01         100.0     -15.0      34.20         16776.107422   
3 2020-01-01         100.0     -15.0      34.32         1660

In [58]:
print(merged_df.isna().sum())

time                   0
pressure_hpa           0
latitude               0
longitude              0
geopotential_height    0
temperature            0
dtype: int64


Doing same for the RH file 

In [59]:
rh_file = next(f for f in files if "RH-100mb_2020010100" in f.name)

print(rh_file)

C:\data2020\RH-100mb_2020010100_ncum_imdaa_reanl_prl_00_100_hpa.nc


In [60]:
rh_ds = xr.open_dataset(rh_file)

print(rh_ds)

<xarray.Dataset> Size: 2MB
Dimensions:  (time: 1, plev: 1, lat: 501, lon: 751)
Coordinates:
  * time     (time) datetime64[ns] 8B 2020-01-01
  * plev     (plev) float64 8B 1e+04
  * lat      (lat) float64 4kB -15.0 -14.88 -14.76 -14.64 ... 44.76 44.88 45.0
  * lon      (lon) float64 6kB 30.0 30.12 30.24 30.36 ... 119.8 119.9 120.0
Data variables:
    r        (time, plev, lat, lon) float32 2MB ...
Attributes:
    CDI:          Climate Data Interface version 2.4.0 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Fri Jun 12 22:47:42 2026: cdo -f nc4c -z zip_4 sellonlatbox...
    CDO:          Climate Data Operators version 2.4.0 (https://mpimet.mpg.de...


In [61]:
rh_df = rh_ds.to_dataframe().reset_index()
rh_df = rh_df.dropna(subset=["r"])
rh_df["plev"] = rh_df["plev"] / 100
rh_df = rh_df.rename(columns={
    "plev": "pressure_hpa",
    "lat": "latitude",
    "lon": "longitude",
    "r": "relative_humidity"
})

In [62]:
merged_df = merged_df.merge(
    rh_df,
    on=["time", "pressure_hpa", "latitude", "longitude"],
    how="inner"
)

In [63]:
print(merged_df.shape)
print(merged_df.info())
print(merged_df.head())

(366720, 7)
<class 'pandas.DataFrame'>
RangeIndex: 366720 entries, 0 to 366719
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   time                 366720 non-null  datetime64[ns]
 1   pressure_hpa         366720 non-null  float64       
 2   latitude             366720 non-null  float64       
 3   longitude            366720 non-null  float64       
 4   geopotential_height  366720 non-null  float32       
 5   temperature          366720 non-null  float32       
 6   relative_humidity    366720 non-null  float32       
dtypes: datetime64[ns](1), float32(3), float64(3)
memory usage: 15.4 MB
None
        time  pressure_hpa  latitude  longitude  geopotential_height  \
0 2020-01-01         100.0     -15.0      33.96         16744.607422   
1 2020-01-01         100.0     -15.0      34.08         16811.607422   
2 2020-01-01         100.0     -15.0      34.20         16776.107422   
3 

Same for UGRD file

In [64]:
ugrd_file = next(f for f in files if "UGRD-100mb_2020010100" in f.name)

print(ugrd_file)

C:\data2020\UGRD-100mb_2020010100_ncum_imdaa_reanl_prl_00_100_hpa.nc


In [65]:
ugrd_ds = xr.open_dataset(ugrd_file)
print(ugrd_ds)

<xarray.Dataset> Size: 2MB
Dimensions:  (time: 1, plev: 1, lat: 501, lon: 751)
Coordinates:
  * time     (time) datetime64[ns] 8B 2020-01-01
  * plev     (plev) float64 8B 1e+04
  * lat      (lat) float64 4kB -15.0 -14.88 -14.76 -14.64 ... 44.76 44.88 45.0
  * lon      (lon) float64 6kB 30.0 30.12 30.24 30.36 ... 119.8 119.9 120.0
Data variables:
    u        (time, plev, lat, lon) float32 2MB ...
Attributes:
    CDI:          Climate Data Interface version 2.4.0 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Fri Jun 12 22:47:16 2026: cdo -f nc4c -z zip_4 sellonlatbox...
    CDO:          Climate Data Operators version 2.4.0 (https://mpimet.mpg.de...


In [66]:
print(ugrd_ds.data_vars)
print(ugrd_ds["u"].attrs)   # if the variable name is 'u'

Data variables:
    u        (time, plev, lat, lon) float32 2MB ...
{'standard_name': 'eastward_wind', 'long_name': 'U component of wind', 'units': 'm s**-1', 'param': '2.2.0'}


In [67]:
ugrd_df = ugrd_ds.to_dataframe().reset_index()

In [68]:
ugrd_df = ugrd_df.dropna(subset=["u"])

In [69]:
ugrd_df["plev"] = ugrd_df["plev"] / 100

In [70]:
ugrd_df = ugrd_df.rename(columns={
    "plev": "pressure_hpa",
    "lat": "latitude",
    "lon": "longitude",
    "u": "u_wind"
})

In [71]:
print(ugrd_df.head())

         time  pressure_hpa  latitude  longitude     u_wind
33 2020-01-01         100.0     -15.0      33.96 -11.260151
34 2020-01-01         100.0     -15.0      34.08 -11.166401
35 2020-01-01         100.0     -15.0      34.20 -10.885151
36 2020-01-01         100.0     -15.0      34.32 -10.197651
37 2020-01-01         100.0     -15.0      34.44  -9.791401


In [72]:
merged_df = merged_df.merge(
    ugrd_df,
    on=["time", "pressure_hpa", "latitude", "longitude"],
    how="inner"
)

In [73]:
print(merged_df.shape)
print(merged_df.info())
print(merged_df.head())

(366601, 8)
<class 'pandas.DataFrame'>
RangeIndex: 366601 entries, 0 to 366600
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   time                 366601 non-null  datetime64[ns]
 1   pressure_hpa         366601 non-null  float64       
 2   latitude             366601 non-null  float64       
 3   longitude            366601 non-null  float64       
 4   geopotential_height  366601 non-null  float32       
 5   temperature          366601 non-null  float32       
 6   relative_humidity    366601 non-null  float32       
 7   u_wind               366601 non-null  float32       
dtypes: datetime64[ns](1), float32(4), float64(3)
memory usage: 16.8 MB
None
        time  pressure_hpa  latitude  longitude  geopotential_height  \
0 2020-01-01         100.0     -15.0      33.96         16744.607422   
1 2020-01-01         100.0     -15.0      34.08         16811.607422   
2 2020-01-01    

Same for VGRD file

In [74]:
vgrd_file = next(f for f in files if "VGRD-100mb_2020010100" in f.name)

print(vgrd_file)

C:\data2020\VGRD-100mb_2020010100_ncum_imdaa_reanl_prl_00_100_hpa.nc


In [75]:
vgrd_ds = xr.open_dataset(vgrd_file)
print(vgrd_ds)

<xarray.Dataset> Size: 2MB
Dimensions:  (time: 1, plev: 1, lat: 501, lon: 751)
Coordinates:
  * time     (time) datetime64[ns] 8B 2020-01-01
  * plev     (plev) float64 8B 1e+04
  * lat      (lat) float64 4kB -15.0 -14.88 -14.76 -14.64 ... 44.76 44.88 45.0
  * lon      (lon) float64 6kB 30.0 30.12 30.24 30.36 ... 119.8 119.9 120.0
Data variables:
    v        (time, plev, lat, lon) float32 2MB ...
Attributes:
    CDI:          Climate Data Interface version 2.4.0 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Fri Jun 12 22:47:23 2026: cdo -f nc4c -z zip_4 sellonlatbox...
    CDO:          Climate Data Operators version 2.4.0 (https://mpimet.mpg.de...


In [76]:
print(vgrd_ds.data_vars)

Data variables:
    v        (time, plev, lat, lon) float32 2MB ...


In [77]:
print(vgrd_ds["v"].attrs)

{'standard_name': 'northward_wind', 'long_name': 'V component of wind', 'units': 'm s**-1', 'param': '3.2.0'}


In [78]:
vgrd_df = vgrd_ds.to_dataframe().reset_index()

In [79]:
vgrd_df = vgrd_df.dropna(subset=["v"])

In [80]:
vgrd_df["plev"] = vgrd_df["plev"] / 100

In [81]:
vgrd_df = vgrd_df.rename(columns={
    "plev": "pressure_hpa",
    "lat": "latitude",
    "lon": "longitude",
    "v": "v_wind"
})

In [82]:
merged_df = merged_df.merge(
    vgrd_df,
    on=["time", "pressure_hpa", "latitude", "longitude"],
    how="inner"
)

In [84]:
print(merged_df.shape)
print(merged_df.info())
print(merged_df.head())

(366601, 9)
<class 'pandas.DataFrame'>
RangeIndex: 366601 entries, 0 to 366600
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   time                 366601 non-null  datetime64[ns]
 1   pressure_hpa         366601 non-null  float64       
 2   latitude             366601 non-null  float64       
 3   longitude            366601 non-null  float64       
 4   geopotential_height  366601 non-null  float32       
 5   temperature          366601 non-null  float32       
 6   relative_humidity    366601 non-null  float32       
 7   u_wind               366601 non-null  float32       
 8   v_wind               366601 non-null  float32       
dtypes: datetime64[ns](1), float32(5), float64(3)
memory usage: 18.2 MB
None
        time  pressure_hpa  latitude  longitude  geopotential_height  \
0 2020-01-01         100.0     -15.0      33.96         16744.607422   
1 2020-01-01         100.0    

Merged Data Frame for one time stamp 

In [85]:
print(merged_df.head())

        time  pressure_hpa  latitude  longitude  geopotential_height  \
0 2020-01-01         100.0     -15.0      33.96         16744.607422   
1 2020-01-01         100.0     -15.0      34.08         16811.607422   
2 2020-01-01         100.0     -15.0      34.20         16776.107422   
3 2020-01-01         100.0     -15.0      34.32         16604.107422   
4 2020-01-01         100.0     -15.0      34.44         16501.607422   

   temperature  relative_humidity     u_wind    v_wind  
0   190.821976          55.289185 -11.260151  5.275614  
1   191.275101          51.101685 -11.166401  5.041239  
2   190.993851          53.757935 -10.885151  4.963114  
3   189.821976          65.570435 -10.197651  4.931864  
4   189.243851          72.164185  -9.791401  4.619364  


Since doing this for all the files will be difficult 

Making Functions to process same things for all the files and saving files for each time stamp in one folder named processed 2020 (ssaving files in paraquent form)

In [86]:
# function to create data frame from netcdf file
import xarray as xr

def preprocess_variable(file_path, variable_name, output_column):
    """
    Preprocess a single NetCDF variable file.

    Parameters
    ----------
    file_path : Path
        Path to the NetCDF file.

    variable_name : str
        Variable inside the file (gh, t, r, u, v).

    output_column : str
        New name for the variable column.

    Returns
    -------
    pandas.DataFrame
    """

    # Open dataset
    ds = xr.open_dataset(file_path)

    # Convert to dataframe
    df = ds.to_dataframe().reset_index()

    # Remove NaN values only from the required variable
    df = df.dropna(subset=[variable_name])

    # Convert pressure from Pa to hPa
    df["plev"] = df["plev"] / 100

    # Rename columns
    df = df.rename(columns={
        "plev": "pressure_hpa",
        "lat": "latitude",
        "lon": "longitude",
        variable_name: output_column
    })

    return df

In [ ]:
# For test whether preprocess_variable() works correctly 
hgt_file = next(f for f in files if "HGT-100mb_2020010100" in f.name)
hgt_df = preprocess_variable(
    hgt_file,
    "gh",
    "geopotential_height"
)

hgt_df.head()

,time,pressure_hpa,latitude,longitude,geopotential_height
33,2020-01-01,100.0,-15.0,33.96,16744.607422
34,2020-01-01,100.0,-15.0,34.08,16811.607422
35,2020-01-01,100.0,-15.0,34.20,16776.107422
36,2020-01-01,100.0,-15.0,34.32,16604.107422
37,2020-01-01,100.0,-15.0,34.44,16501.607422


In [89]:
tmp_df = preprocess_variable(
    tmp_file,
    "t",
    "temperature"
)

tmp_df.head()

,time,pressure_hpa,latitude,longitude,temperature
33,2020-01-01,100.0,-15.0,33.96,190.821976
34,2020-01-01,100.0,-15.0,34.08,191.275101
35,2020-01-01,100.0,-15.0,34.20,190.993851
36,2020-01-01,100.0,-15.0,34.32,189.821976
37,2020-01-01,100.0,-15.0,34.44,189.243851


In [90]:
rh_df = preprocess_variable(
    rh_file,
    "r",
    "relative_humidity"
)

rh_df.head()

,time,pressure_hpa,latitude,longitude,relative_humidity
33,2020-01-01,100.0,-15.0,33.96,55.289185
34,2020-01-01,100.0,-15.0,34.08,51.101685
35,2020-01-01,100.0,-15.0,34.20,53.757935
36,2020-01-01,100.0,-15.0,34.32,65.570435
37,2020-01-01,100.0,-15.0,34.44,72.164185


In [91]:
ugrd_df = preprocess_variable(
    ugrd_file,
    "u",
    "u_wind"
)

ugrd_df.head()

,time,pressure_hpa,latitude,longitude,u_wind
33,2020-01-01,100.0,-15.0,33.96,-11.260151
34,2020-01-01,100.0,-15.0,34.08,-11.166401
35,2020-01-01,100.0,-15.0,34.20,-10.885151
36,2020-01-01,100.0,-15.0,34.32,-10.197651
37,2020-01-01,100.0,-15.0,34.44,-9.791401


In [92]:
vgrd_df = preprocess_variable(
    vgrd_file,
    "v",
    "v_wind"
)

vgrd_df.head()

,time,pressure_hpa,latitude,longitude,v_wind
33,2020-01-01,100.0,-15.0,33.96,5.275614
34,2020-01-01,100.0,-15.0,34.08,5.041239
35,2020-01-01,100.0,-15.0,34.20,4.963114
36,2020-01-01,100.0,-15.0,34.32,4.931864
37,2020-01-01,100.0,-15.0,34.44,4.619364


In [93]:
print(hgt_df.shape)
print(tmp_df.shape)
print(rh_df.shape)
print(ugrd_df.shape)
print(vgrd_df.shape)

(366720, 5)
(366720, 5)
(366720, 5)
(366738, 5)
(366738, 5)


In [94]:
print(type(files))
print(len(files))

<class 'list'>
88864


In [95]:
# file finder function
def get_file(variable, pressure, timestamp):
    """
    Returns the file path for a given variable,
    pressure level and timestamp.
    """

    pattern = f"{variable}-{pressure}_{timestamp}"

    return next(f for f in files if pattern in f.name)

In [96]:
print(get_file("HGT", "100mb", "2020010100"))

C:\data2020\HGT-100mb_2020010100_ncum_imdaa_reanl_prl_00_100_hpa.nc


In [97]:
# processes entire time stamp and returns a merged dataframe
def process_timestamp(timestamp, pressure):

    # Preprocess all variables
    hgt_df = preprocess_variable(
        get_file("HGT", pressure, timestamp),
        "gh",
        "geopotential_height"
    )

    tmp_df = preprocess_variable(
        get_file("TMP", pressure, timestamp),
        "t",
        "temperature"
    )

    rh_df = preprocess_variable(
        get_file("RH", pressure, timestamp),
        "r",
        "relative_humidity"
    )

    ugrd_df = preprocess_variable(
        get_file("UGRD", pressure, timestamp),
        "u",
        "u_wind"
    )

    vgrd_df = preprocess_variable(
        get_file("VGRD", pressure, timestamp),
        "v",
        "v_wind"
    )

    # Merge everything
    merged = hgt_df

    for df in [tmp_df, rh_df, ugrd_df, vgrd_df]:
        merged = merged.merge(
            df,
            on=["time", "pressure_hpa", "latitude", "longitude"],
            how="inner"
        )

    return merged

In [ ]:
#for checking process_timestamp function
merged_df = process_timestamp("2020010100", "100mb")

In [99]:
print(merged_df.shape)
merged_df.head()

(366601, 9)


,time,pressure_hpa,latitude,longitude,geopotential_height,temperature,relative_humidity,u_wind,v_wind
0,2020-01-01,100.0,-15.0,33.96,16744.607422,190.821976,55.289185,-11.260151,5.275614
1,2020-01-01,100.0,-15.0,34.08,16811.607422,191.275101,51.101685,-11.166401,5.041239
2,2020-01-01,100.0,-15.0,34.20,16776.107422,190.993851,53.757935,-10.885151,4.963114
3,2020-01-01,100.0,-15.0,34.32,16604.107422,189.821976,65.570435,-10.197651,4.931864
4,2020-01-01,100.0,-15.0,34.44,16501.607422,189.243851,72.164185,-9.791401,4.619364


In [100]:
#Extracting all the time stamps automatically
timestamps = sorted({
    file.name.split("_")[1]
    for file in files
    if file.name.startswith("HGT-100mb")
})

print("Total timestamps:", len(timestamps))
print(timestamps[:10])

Total timestamps: 1776
['2020010100', '2020010103', '2020010106', '2020010109', '2020010112', '2020010115', '2020010118', '2020010121', '2020010200', '2020010203']


In [101]:
print("First timestamp :", timestamps[0])
print("Last timestamp  :", timestamps[-1])
print("Total timestamps:", len(timestamps))

First timestamp : 2020010100
Last timestamp  : 2020123121
Total timestamps: 1776


In [102]:
#first 5 timestamps
processed_data = []

for ts in timestamps[:5]:
    print(f"Processing {ts}")

    df = process_timestamp(ts, "100mb")

    processed_data.append(df)

print("Done!")

Processing 2020010100
Processing 2020010103
Processing 2020010106
Processing 2020010109
Processing 2020010112
Done!


In [ ]:
#checking if the merged function works for all timestamps
import pandas as pd
combined_df = pd.concat(processed_data, ignore_index=True)

print(combined_df.shape)
combined_df.head()

(1833005, 9)


,time,pressure_hpa,latitude,longitude,geopotential_height,temperature,relative_humidity,u_wind,v_wind
0,2020-01-01,100.0,-15.0,33.96,16744.607422,190.821976,55.289185,-11.260151,5.275614
1,2020-01-01,100.0,-15.0,34.08,16811.607422,191.275101,51.101685,-11.166401,5.041239
2,2020-01-01,100.0,-15.0,34.20,16776.107422,190.993851,53.757935,-10.885151,4.963114
3,2020-01-01,100.0,-15.0,34.32,16604.107422,189.821976,65.570435,-10.197651,4.931864
4,2020-01-01,100.0,-15.0,34.44,16501.607422,189.243851,72.164185,-9.791401,4.619364


In [105]:
df = process_timestamp("2020010100", "100mb")

In [ ]:
from pathlib import Path

# Create a folder named 'processed_2020' inside your project to save the parquent files 
output_folder = Path("processed_2020")
output_folder.mkdir(exist_ok=True)

print(output_folder)

processed_2020


In [ ]:
#checking if the parquet file is saved correctly
output_file = output_folder / "2020010100.parquet"

df.to_parquet(output_file, index=False)

print("Saved:", output_file)

Saved: processed_2020\2020010100.parquet


In [110]:
test_df = pd.read_parquet(output_file)

print(test_df.shape)
test_df.head()

(366601, 9)


,time,pressure_hpa,latitude,longitude,geopotential_height,temperature,relative_humidity,u_wind,v_wind
0,2020-01-01,100.0,-15.0,33.96,16744.607422,190.821976,55.289185,-11.260151,5.275614
1,2020-01-01,100.0,-15.0,34.08,16811.607422,191.275101,51.101685,-11.166401,5.041239
2,2020-01-01,100.0,-15.0,34.20,16776.107422,190.993851,53.757935,-10.885151,4.963114
3,2020-01-01,100.0,-15.0,34.32,16604.107422,189.821976,65.570435,-10.197651,4.931864
4,2020-01-01,100.0,-15.0,34.44,16501.607422,189.243851,72.164185,-9.791401,4.619364


In [ ]:
# Process all timestamps and save the preprocessed data as Parquet files
from pathlib import Path

output_folder = Path("processed_2020")
output_folder.mkdir(exist_ok=True)

failed = []

for i, ts in enumerate(timestamps):

    output_file = output_folder / f"{ts}.parquet"

    # Skip timestamps already processed
    if output_file.exists():
        print(f"[{i+1}/{len(timestamps)}] Skipping {ts}")
        continue

    print(f"[{i+1}/{len(timestamps)}] Processing {ts}")

    try:
        df = process_all_pressures(ts)
        df.to_parquet(output_file, index=False)

    except Exception as e:
        print(f"Failed: {ts}")
        print(e)
        failed.append((ts, str(e)))

print("Processing completed!")
print("Failed files:", len(failed))

[1/1776] Processing 2020010100
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
[2/1776] Processing 2020010103
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
[3/1776] Processing 2020010106
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
[4/1776] Processing 2020010109
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
[5/1776] Processing 2020010112
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
[6/1776] Processing 2020010115
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
[7/1776] Processing 2020010118
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
[8/1776] Processing 2020010121
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processi

In [ ]:
# Create a directory to store the processed Parquet files
from pathlib import Path

output_folder = Path("processed_2020")
output_folder.mkdir(exist_ok=True)

In [112]:
pressure_levels = sorted({
    file.name.split("_")[-1].replace("_hpa.nc", "")
    for file in files
    if file.name.startswith("HGT-")
})

print(pressure_levels)
print("Total pressure levels:", len(pressure_levels))

['hpa.nc']
Total pressure levels: 1


In [113]:
print(files[0].name)
print(files[100].name)
print(files[1000].name)

HGT-100mb_2020010100_ncum_imdaa_reanl_prl_00_100_hpa.nc
HGT-100mb_2020011312_ncum_imdaa_reanl_prl_12_100_hpa.nc
HGT-100mb_2020050500_ncum_imdaa_reanl_prl_00_100_hpa.nc


In [ ]:
# Pressure levels selected for preprocessing
pressure_levels = [
    "50mb",
    "70mb",
    "100mb",
    "150mb",
    "200mb"
]

print(pressure_levels)
print("Total pressure levels:", len(pressure_levels))

['50mb', '70mb', '100mb', '150mb', '200mb']
Total pressure levels: 5


In [ ]:
"""
    Processes all selected pressure levels for a given timestamp.

    Parameters:
        timestamp (str): Timestamp in YYYYMMDDHH format.

    Returns:
        pandas.DataFrame: Combined dataset containing all selected
        pressure levels for the given timestamp.
"""

import pandas as pd

def process_all_pressures(timestamp):

    all_data = []

    for pressure in pressure_levels:

        print(f"Processing {pressure}...")

        try:
            df = process_timestamp(timestamp, pressure)
            all_data.append(df)

        except Exception as e:
            print(f"Skipped {pressure}: {e}")

    combined = pd.concat(all_data, ignore_index=True)

    return combined

In [ ]:
#a test to check whether the function process_all_pressures() works correctly for a single timestamp.
df = process_all_pressures("2020010100")

Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...


In [ ]:
# verifying process_all_pressures() returned a DataFrame.
print(df.shape)

df.head()

(3299409, 9)


,time,pressure_hpa,latitude,longitude,geopotential_height,temperature,relative_humidity,u_wind,v_wind
0,2020-01-01,100.0,-15.0,33.96,16744.607422,190.821976,55.289185,-11.260151,5.275614
1,2020-01-01,100.0,-15.0,34.08,16811.607422,191.275101,51.101685,-11.166401,5.041239
2,2020-01-01,100.0,-15.0,34.20,16776.107422,190.993851,53.757935,-10.885151,4.963114
3,2020-01-01,100.0,-15.0,34.32,16604.107422,189.821976,65.570435,-10.197651,4.931864
4,2020-01-01,100.0,-15.0,34.44,16501.607422,189.243851,72.164185,-9.791401,4.619364


In [ ]:
# Verify the pressure levels in the processed file
print(df["pressure_hpa"].unique())

[ 50.  70. 100. 150. 200.]


In [ ]:
# testing that get_file() is returning the correct file paths.
for p in pressure_levels:
    try:
        print(p, get_file("HGT", p, "2020010100"))
    except Exception as e:
        print(f"{p} -> {e}")

100mb C:\data2020\HGT-100mb_2020010100_ncum_imdaa_reanl_prl_00_100_hpa.nc
150mb C:\data2020\HGT-150mb_2020010100_ncum_imdaa_reanl_prl_00_150_hpa.nc
200mb C:\data2020\HGT-200mb_2020010100_ncum_imdaa_reanl_prl_00_200_hpa.nc
250mb C:\data2020\HGT-250mb_2020010100_ncum_imdaa_reanl_prl_00_50_hpa.nc
50mb C:\data2020\HGT-50mb_2020010100_ncum_imdaa_reanl_prl_00_50_hpa.nc
70mb C:\data2020\HGT-70mb_2020010100_ncum_imdaa_reanl_prl_00_70_hpa.nc
750mb C:\data2020\HGT-750mb_2020010100_ncum_imdaa_reanl_prl_00_50_hpa.nc
850mb C:\data2020\HGT-850mb_2020010100_ncum_imdaa_reanl_prl_00_50_hpa.nc
950mb C:\data2020\HGT-950mb_2020010100_ncum_imdaa_reanl_prl_00_50_hpa.nc


In [ ]:
#for detectin the actual file name for the pressure level
print(get_file("HGT", "250mb", "2020010100").name)
print(get_file("HGT", "750mb", "2020010100").name)
print(get_file("HGT", "850mb", "2020010100").name)
print(get_file("HGT", "950mb", "2020010100").name)

HGT-250mb_2020010100_ncum_imdaa_reanl_prl_00_50_hpa.nc
HGT-750mb_2020010100_ncum_imdaa_reanl_prl_00_50_hpa.nc
HGT-850mb_2020010100_ncum_imdaa_reanl_prl_00_50_hpa.nc
HGT-950mb_2020010100_ncum_imdaa_reanl_prl_00_50_hpa.nc


In [ ]:
# Verify Pressure Level Metadata
# ----------------------------

# Open a sample NetCDF file to verify that the pressure level
# stored in the dataset matches the expected pressure level.
import xarray as xr

file = get_file("HGT", "250mb", "2020010100")

ds = xr.open_dataset(file)

print(ds)


<xarray.Dataset> Size: 2MB
Dimensions:  (time: 1, plev: 1, lat: 501, lon: 751)
Coordinates:
  * time     (time) datetime64[ns] 8B 2020-01-01
  * plev     (plev) float64 8B 2.5e+04
  * lat      (lat) float64 4kB -15.0 -14.88 -14.76 -14.64 ... 44.76 44.88 45.0
  * lon      (lon) float64 6kB 30.0 30.12 30.24 30.36 ... 119.8 119.9 120.0
Data variables:
    gh       (time, plev, lat, lon) float32 2MB ...
Attributes:
    CDI:          Climate Data Interface version 2.4.0 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Fri Jun 12 22:47:29 2026: cdo -f nc4c -z zip_4 sellonlatbox...
    CDO:          Climate Data Operators version 2.4.0 (https://mpimet.mpg.de...


In [ ]:
# Verify the actual pressure level from the NetCDF metadata
print(ds["plev"].values)

[25000.]


In [ ]:
# Process all selected pressure levels for a sample timestamp
df = process_all_pressures("2020010100")

Processing 100mb...
Processing 150mb...
Processing 200mb...
Processing 250mb...
Processing 50mb...
Processing 70mb...
Processing 750mb...
Processing 850mb...
Processing 950mb...


In [126]:
print(df.shape)

(3299409, 9)


In [127]:
print(sorted(df["pressure_hpa"].unique()))

[np.float64(50.0), np.float64(70.0), np.float64(100.0), np.float64(150.0), np.float64(200.0), np.float64(250.0), np.float64(750.0), np.float64(850.0), np.float64(950.0)]


Checking the paraquent files if all files are processed or not and if they are processed correctly or not 

In [136]:
#checkin the procesing of the paraquent files are done correctly
from pathlib import Path

output_folder = Path("processed_2020")

parquet_files = list(output_folder.glob("*.parquet"))

print("Total Parquet files:", len(parquet_files))

Total Parquet files: 1776


In [137]:
#checking 1st paraquet file
import pandas as pd

df = pd.read_parquet(parquet_files[0])

print(df.head())
print(df.info())

        time  pressure_hpa  latitude  longitude  geopotential_height  \
0 2020-01-01          50.0     -15.0      33.96         20731.246094   
1 2020-01-01          50.0     -15.0      34.08         20819.746094   
2 2020-01-01          50.0     -15.0      34.20         20772.246094   
3 2020-01-01          50.0     -15.0      34.32         20539.246094   
4 2020-01-01          50.0     -15.0      34.44         20399.246094   

   temperature  relative_humidity    u_wind    v_wind  
0   208.090027           3.403575 -5.746175 -0.919383  
1   209.363464           2.856700 -5.824300 -0.216258  
2   208.793152           3.122325 -5.386800 -0.935008  
3   205.754089           4.716075 -3.996175 -2.966258  
4   203.941589           6.075450 -2.168050 -4.638133  
<class 'pandas.DataFrame'>
RangeIndex: 1833005 entries, 0 to 1833004
Data columns (total 9 columns):
 #   Column               Dtype         
---  ------               -----         
 0   time                 datetime64[ns]
 1   pr

In [138]:
#checking pressure levels
print(sorted(df["pressure_hpa"].unique()))

[np.float64(50.0), np.float64(70.0), np.float64(100.0), np.float64(150.0), np.float64(200.0)]


In [139]:
#checking for null values
print(df.isnull().sum())

time                   0
pressure_hpa           0
latitude               0
longitude              0
geopotential_height    0
temperature            0
relative_humidity      0
u_wind                 0
v_wind                 0
dtype: int64


In [140]:
#checking time stamps(each file should have a unique timestamp )
print(df["time"].unique())

<DatetimeArray>
['2020-01-01 00:00:00']
Length: 1, dtype: datetime64[ns]


# ============================================================
#                2019 DATA PREPROCESSING
# ============================================================

In [2]:
# Import Libraries

from pathlib import Path

import pandas as pd
import xarray as xr

In [7]:
# Step 2: Load 2019 Dataset

data_path = Path("C:/data2019")  

files = sorted(list(data_path.glob("*.nc")))

print("Total NetCDF files:", len(files))
print("Sample file:", files[0].name)

Total NetCDF files: 146000
Sample file: HGT-100mb_2019010100_ncum_imdaa_reanl_prl_00_100_hpa.nc


In [22]:
# Step 3: Extract Timestamps


timestamps = sorted({
    file.name.split("_")[1]
    for file in files
})

print("Total timestamps:", len(timestamps))
print("First timestamp:", timestamps[0])
print("Last timestamp:", timestamps[-1])

Total timestamps: 2920
First timestamp: 2019010100
Last timestamp: 2019123121


In [23]:
# Step 4: Pressure Levels

pressure_levels = [
    "50mb",
    "70mb",
    "100mb",
    "150mb",
    "200mb"
]

print(pressure_levels)

['50mb', '70mb', '100mb', '150mb', '200mb']


In [24]:
# Step 5: File Finder


def get_file(variable, pressure, timestamp):
    """
    Returns the NetCDF file for the given
    variable, pressure level and timestamp.
    """

    pattern = f"{variable}-{pressure}_{timestamp}"

    try:
        return next(f for f in files if pattern in f.name)

    except StopIteration:
        raise FileNotFoundError(
            f"{variable} {pressure} {timestamp} not found"
        )

In [25]:
# verifying the get_file() function for a sample timestamp and pressure level
for pressure in pressure_levels:
    df = process_timestamp("2019010100", pressure)
    print(pressure, df["pressure_hpa"].unique())

50mb [50.]
70mb [70.]
100mb [100.]
150mb [150.]
200mb [200.]


In [26]:
# Step 6: Variable Preprocessing

def preprocess_variable(file_path, variable_name, output_column):
    """
    Converts a NetCDF variable into a cleaned DataFrame.
    """

    ds = xr.open_dataset(file_path)

    df = ds.to_dataframe().reset_index()

    ds.close()

    df = df.dropna(subset=[variable_name])

    df["plev"] = df["plev"] / 100

    df = df.rename(columns={
        "plev": "pressure_hpa",
        "lat": "latitude",
        "lon": "longitude",
        variable_name: output_column
    })

    return df

In [27]:
print(df.shape)
df.head()

(376251, 6)


,time,plev,lat,lon,gh,pressure_hpa
0,2019010100,20000.0,-15.0,30.00,NaN,200.0
1,2019010100,20000.0,-15.0,30.12,NaN,200.0
2,2019010100,20000.0,-15.0,30.24,NaN,200.0
3,2019010100,20000.0,-15.0,30.36,NaN,200.0
4,2019010100,20000.0,-15.0,30.48,NaN,200.0


In [28]:
# Step 7: Process One Timestamp


def process_timestamp(timestamp, pressure):

    hgt_df = preprocess_variable(
        get_file("HGT", pressure, timestamp),
        "gh",
        "geopotential_height"
    )

    tmp_df = preprocess_variable(
        get_file("TMP", pressure, timestamp),
        "t",
        "temperature"
    )

    rh_df = preprocess_variable(
        get_file("RH", pressure, timestamp),
        "r",
        "relative_humidity"
    )

    ugrd_df = preprocess_variable(
        get_file("UGRD", pressure, timestamp),
        "u",
        "u_wind"
    )

    vgrd_df = preprocess_variable(
        get_file("VGRD", pressure, timestamp),
        "v",
        "v_wind"
    )

    merged = hgt_df

    for df in [tmp_df, rh_df, ugrd_df, vgrd_df]:

        merged = merged.merge(
            df,
            on=[
                "time",
                "pressure_hpa",
                "latitude",
                "longitude"
            ],
            how="inner"
        )

    return merged

In [30]:
# Step 8: Process All Pressures

def process_all_pressures(timestamp):

    all_data = []

    for pressure in pressure_levels:

        print(f"Processing {pressure}...")

        try:

            df = process_timestamp(timestamp, pressure)

            all_data.append(df)

        except Exception as e:

            print(f"Skipped {pressure}: {e}")

    if not all_data:
        raise ValueError(
            f"No data found for {timestamp}"
        )

    combined = pd.concat(
        all_data,
        ignore_index=True
    )

    return combined

In [31]:
# Step 9: Verify One Timestamp

df = process_all_pressures("2019010100")

print(df.shape)

print(df.columns)

print(df.isnull().sum())

print(sorted(df["pressure_hpa"].unique()))

Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
(1833005, 9)
Index(['time', 'pressure_hpa', 'latitude', 'longitude', 'geopotential_height',
       'temperature', 'relative_humidity', 'u_wind', 'v_wind'],
      dtype='str')
time                   0
pressure_hpa           0
latitude               0
longitude              0
geopotential_height    0
temperature            0
relative_humidity      0
u_wind                 0
v_wind                 0
dtype: int64
[np.float64(50.0), np.float64(70.0), np.float64(100.0), np.float64(150.0), np.float64(200.0)]


In [32]:
# Step 10: Output Folder

output_folder = Path("processed_2019")

output_folder.mkdir(exist_ok=True)

print(output_folder)

processed_2019


In [42]:
# Step 11: Process Entire Dataset

failed = []

for i, ts in enumerate(timestamps):

    output_file = output_folder / f"{ts}.parquet"

    if output_file.exists():

        print(f"[{i+1}/{len(timestamps)}] Skipping {ts}")

        continue

    print(f"[{i+1}/{len(timestamps)}] Processing {ts}")

    try:

        df = process_all_pressures(ts)

        if df.empty:
            raise ValueError("Empty DataFrame")

        df.to_parquet(
            output_file,
            index=False
        )

    except Exception as e:

        print(f"Failed: {ts}")

        print(e)

        failed.append((ts, str(e)))

print("\nProcessing Completed!")

print("Failed timestamps:", len(failed))

[1/2920] Skipping 2019010100
[2/2920] Skipping 2019010103
[3/2920] Skipping 2019010106
[4/2920] Skipping 2019010109
[5/2920] Skipping 2019010112
[6/2920] Processing 2019010115
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
[7/2920] Processing 2019010118
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
[8/2920] Processing 2019010121
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
[9/2920] Processing 2019010200
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
[10/2920] Processing 2019010203
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
[11/2920] Processing 2019010206
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
[12/2920] Processing 2019010209
Processing 50mb...
Processing 70mb...
Processin

In [33]:
#testing some code
#Test 1: One pressure level
df = process_timestamp("2019010100", "100mb")

print(df.shape)
print(df.head())

print(df.isnull().sum())
print(df["pressure_hpa"].unique())

(366601, 9)
        time  pressure_hpa  latitude  longitude  geopotential_height  \
0 2019-01-01         100.0     -15.0      33.96          16510.09375   
1 2019-01-01         100.0     -15.0      34.08          16567.59375   
2 2019-01-01         100.0     -15.0      34.20          16634.59375   
3 2019-01-01         100.0     -15.0      34.32          16705.09375   
4 2019-01-01         100.0     -15.0      34.44          16704.59375   

   temperature  relative_humidity    u_wind    v_wind  
0   192.331726          59.583565 -4.378164 -1.249575  
1   192.777039          55.052315 -4.409414 -1.530825  
2   193.253601          50.239815 -4.346914 -1.843325  
3   193.730164          45.333565 -4.190664 -2.093325  
4   193.769226          45.271065 -3.753164 -2.077700  
time                   0
pressure_hpa           0
latitude               0
longitude              0
geopotential_height    0
temperature            0
relative_humidity      0
u_wind                 0
v_wind             

In [34]:
#Test 2: One complete timestamp
df = process_all_pressures("2019010100")

print(df.shape)

print(df.columns.tolist())

print(df["pressure_hpa"].unique())

print(df.isnull().sum())

Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
(1833005, 9)
['time', 'pressure_hpa', 'latitude', 'longitude', 'geopotential_height', 'temperature', 'relative_humidity', 'u_wind', 'v_wind']
[ 50.  70. 100. 150. 200.]
time                   0
pressure_hpa           0
latitude               0
longitude              0
geopotential_height    0
temperature            0
relative_humidity      0
u_wind                 0
v_wind                 0
dtype: int64


In [35]:
#Test 3: Save one Parquet file
output_file = output_folder / "2019010100.parquet"

df.to_parquet(output_file, index=False)

print("Saved:", output_file)

Saved: processed_2019\2019010100.parquet


In [36]:
#Test 4: Read it back
test_df = pd.read_parquet(output_file)

print(test_df.shape)

print(test_df.head())

print(test_df.isnull().sum())

print(sorted(test_df["pressure_hpa"].unique()))

(1833005, 9)
        time  pressure_hpa  latitude  longitude  geopotential_height  \
0 2019-01-01          50.0     -15.0      33.96         20456.568359   
1 2019-01-01          50.0     -15.0      34.08         20534.068359   
2 2019-01-01          50.0     -15.0      34.20         20623.568359   
3 2019-01-01          50.0     -15.0      34.32         20716.568359   
4 2019-01-01          50.0     -15.0      34.44         20715.068359   

   temperature  relative_humidity     u_wind    v_wind  
0   202.932434           4.554688 -17.626209  0.869842  
1   203.994934           3.902344 -17.485584  0.885467  
2   205.182434           3.277344 -17.516834  1.033904  
3   206.338684           2.769531 -17.501209  1.174529  
4   206.190247           2.855469 -16.782459  1.315154  
time                   0
pressure_hpa           0
latitude               0
longitude              0
geopotential_height    0
temperature            0
relative_humidity      0
u_wind                 0
v_wind      

In [37]:
#Test 5: Process only 5 timestamps
#Before running step 11 we will first test the processing of only 5 timestamps to ensure everything works correctly.
failed = []

for ts in timestamps[:5]:

    print(f"\nProcessing {ts}")

    try:
        df = process_all_pressures(ts)

        output_file = output_folder / f"{ts}.parquet"

        df.to_parquet(output_file, index=False)

        print("✔ Saved")

    except Exception as e:
        print("❌ Failed:", e)
        failed.append(ts)

print("\nFailed:", failed)


Processing 2019010100
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
✔ Saved

Processing 2019010103
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
✔ Saved

Processing 2019010106
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
✔ Saved

Processing 2019010109
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
✔ Saved

Processing 2019010112
Processing 50mb...
Processing 70mb...
Processing 100mb...
Processing 150mb...
Processing 200mb...
✔ Saved

Failed: []


In [38]:
#Test 6: Verify the Parquet files created
parquet_files = list(output_folder.glob("*.parquet"))

print("Files created:", len(parquet_files))

for f in parquet_files:
    print(f.name)

Files created: 5
2019010100.parquet
2019010103.parquet
2019010106.parquet
2019010109.parquet
2019010112.parquet


In [39]:
#Open one of the Parquet files to verify its contents
import pandas as pd

df = pd.read_parquet(parquet_files[0])

print(df.shape)
print(df.head())

(1833005, 9)
        time  pressure_hpa  latitude  longitude  geopotential_height  \
0 2019-01-01          50.0     -15.0      33.96         20456.568359   
1 2019-01-01          50.0     -15.0      34.08         20534.068359   
2 2019-01-01          50.0     -15.0      34.20         20623.568359   
3 2019-01-01          50.0     -15.0      34.32         20716.568359   
4 2019-01-01          50.0     -15.0      34.44         20715.068359   

   temperature  relative_humidity     u_wind    v_wind  
0   202.932434           4.554688 -17.626209  0.869842  
1   203.994934           3.902344 -17.485584  0.885467  
2   205.182434           3.277344 -17.516834  1.033904  
3   206.338684           2.769531 -17.501209  1.174529  
4   206.190247           2.855469 -16.782459  1.315154  


In [40]:
print(df.isnull().sum())

time                   0
pressure_hpa           0
latitude               0
longitude              0
geopotential_height    0
temperature            0
relative_humidity      0
u_wind                 0
v_wind                 0
dtype: int64


In [41]:
print(sorted(df["pressure_hpa"].unique()))

[np.float64(50.0), np.float64(70.0), np.float64(100.0), np.float64(150.0), np.float64(200.0)]


In [44]:
# Done with step 11 
#checking the parquet files created for all timestamps
from pathlib import Path

processed_2019 = Path("processed_2019")
processed_2020 = Path("processed_2020")

files_2019 = sorted(processed_2019.glob("*.parquet"))
files_2020 = sorted(processed_2020.glob("*.parquet"))

print("2019 files:", len(files_2019))
print("2020 files:", len(files_2020))

2019 files: 2920
2020 files: 1776


In [45]:
#Check 2: Read a few random files
import pandas as pd
import random

sample_files = random.sample(files_2019, 5)

for file in sample_files:
    df = pd.read_parquet(file)

    print("\n", file.name)
    print(df.shape)


 2019032018.parquet
(1833005, 9)

 2019051815.parquet
(1833005, 9)

 2019080412.parquet
(1833005, 9)

 2019050903.parquet
(1833005, 9)

 2019111715.parquet
(1833005, 9)


In [46]:
#Check 3: Verify columns
expected_columns = [
    "time",
    "pressure_hpa",
    "latitude",
    "longitude",
    "geopotential_height",
    "temperature",
    "relative_humidity",
    "u_wind",
    "v_wind"
]

for file in sample_files:
    df = pd.read_parquet(file)

    print(file.name)

    if list(df.columns) == expected_columns:
        print("✅ Columns correct")
    else:
        print("❌ Column mismatch")
        print(df.columns.tolist())

2019032018.parquet
✅ Columns correct
2019051815.parquet
✅ Columns correct
2019080412.parquet
✅ Columns correct
2019050903.parquet
✅ Columns correct
2019111715.parquet
✅ Columns correct


In [47]:
#Check 4: Missing values
for file in sample_files:

    df = pd.read_parquet(file)

    print(file.name)

    print(df.isnull().sum())

    print("-"*40)

2019032018.parquet
time                   0
pressure_hpa           0
latitude               0
longitude              0
geopotential_height    0
temperature            0
relative_humidity      0
u_wind                 0
v_wind                 0
dtype: int64
----------------------------------------
2019051815.parquet
time                   0
pressure_hpa           0
latitude               0
longitude              0
geopotential_height    0
temperature            0
relative_humidity      0
u_wind                 0
v_wind                 0
dtype: int64
----------------------------------------
2019080412.parquet
time                   0
pressure_hpa           0
latitude               0
longitude              0
geopotential_height    0
temperature            0
relative_humidity      0
u_wind                 0
v_wind                 0
dtype: int64
----------------------------------------
2019050903.parquet
time                   0
pressure_hpa           0
latitude               0
longitude   

In [48]:
#Check 5: Pressure levels
for file in sample_files:

    df = pd.read_parquet(file)

    print(file.name)

    print(sorted(df["pressure_hpa"].unique()))

2019032018.parquet
[np.float64(50.0), np.float64(70.0), np.float64(100.0), np.float64(150.0), np.float64(200.0)]
2019051815.parquet
[np.float64(50.0), np.float64(70.0), np.float64(100.0), np.float64(150.0), np.float64(200.0)]
2019080412.parquet
[np.float64(50.0), np.float64(70.0), np.float64(100.0), np.float64(150.0), np.float64(200.0)]
2019050903.parquet
[np.float64(50.0), np.float64(70.0), np.float64(100.0), np.float64(150.0), np.float64(200.0)]
2019111715.parquet
[np.float64(50.0), np.float64(70.0), np.float64(100.0), np.float64(150.0), np.float64(200.0)]


In [49]:
#Check 6: Duplicate rows
for file in sample_files:

    df = pd.read_parquet(file)

    duplicates = df.duplicated().sum()

    print(file.name, duplicates)

2019032018.parquet 0
2019051815.parquet 0
2019080412.parquet 0
2019050903.parquet 0
2019111715.parquet 0


In [50]:
#Check 7: Data types
df = pd.read_parquet(files_2019[0])

print(df.dtypes)

time                   datetime64[ns]
pressure_hpa                  float64
latitude                      float64
longitude                     float64
geopotential_height           float32
temperature                   float32
relative_humidity             float32
u_wind                        float32
v_wind                        float32
dtype: object


In [51]:
#Check 8: Basic statistics
df.describe()

,time,pressure_hpa,latitude,longitude,geopotential_height,temperature,relative_humidity,u_wind,v_wind
count,1833005,1.833005e+06,1.833005e+06,1.833005e+06,1.833005e+06,1.833005e+06,1.833005e+06,1.833005e+06,1.833005e+06
mean,2019-01-01 00:00:00,1.140000e+02,1.553214e+01,7.456668e+01,1.634250e+04,2.058678e+02,2.044822e+01,1.046923e+01,7.043884e-01
min,2019-01-01 00:00:00,5.000000e+01,-1.500000e+01,3.000000e+01,1.114971e+04,1.875270e+02,5.000000e-01,-3.462500e+01,-2.000000e+01
25%,2019-01-01 00:00:00,7.000000e+01,8.400000e-01,5.256000e+01,1.387113e+04,1.972302e+02,4.375000e+00,-6.031250e+00,-2.875000e+00
50%,2019-01-01 00:00:00,1.000000e+02,1.572000e+01,7.452000e+01,1.658609e+04,2.058750e+02,1.415625e+01,8.430685e+00,1.233130e-01
75%,2019-01-01 00:00:00,1.500000e+02,3.036000e+01,9.648000e+01,1.859091e+04,2.141406e+02,3.456308e+01,2.377444e+01,3.281675e+00
max,2019-01-01 00:00:00,2.000000e+02,4.500000e+01,1.200000e+02,2.104157e+04,2.243711e+02,8.749278e+01,8.477444e+01,3.400000e+01
std,NaN,5.462602e+01,1.721468e+01,2.540684e+01,2.973541e+03,9.851536e+00,1.807113e+01,1.989294e+01,6.265680e+00


In [52]:
#Check 9: Verify one timestamp
df = pd.read_parquet(files_2019[0])

print(df.head())

print(df.tail())

        time  pressure_hpa  latitude  longitude  geopotential_height  \
0 2019-01-01          50.0     -15.0      33.96         20456.568359   
1 2019-01-01          50.0     -15.0      34.08         20534.068359   
2 2019-01-01          50.0     -15.0      34.20         20623.568359   
3 2019-01-01          50.0     -15.0      34.32         20716.568359   
4 2019-01-01          50.0     -15.0      34.44         20715.068359   

   temperature  relative_humidity     u_wind    v_wind  
0   202.932434           4.554688 -17.626209  0.869842  
1   203.994934           3.902344 -17.485584  0.885467  
2   205.182434           3.277344 -17.516834  1.033904  
3   206.338684           2.769531 -17.501209  1.174529  
4   206.190247           2.855469 -16.782459  1.315154  
              time  pressure_hpa  latitude  longitude  geopotential_height  \
1833000 2019-01-01         200.0      45.0     117.60         11320.710938   
1833001 2019-01-01         200.0      45.0     117.72         11307.7

In [53]:
#Check 10: Compare 2019 and 2020
df2019 = pd.read_parquet(files_2019[0])
df2020 = pd.read_parquet(files_2020[0])

print(df2019.columns.equals(df2020.columns))

True


In [54]:
df2019 = pd.read_parquet(sorted(processed_2019.glob("*.parquet"))[0])
df2020 = pd.read_parquet(sorted(processed_2020.glob("*.parquet"))[0])

print(df2019.columns.equals(df2020.columns))
print(df2019.dtypes)
print(df2020.dtypes)
print(df2019.shape)
print(df2020.shape)

True
time                   datetime64[ns]
pressure_hpa                  float64
latitude                      float64
longitude                     float64
geopotential_height           float32
temperature                   float32
relative_humidity             float32
u_wind                        float32
v_wind                        float32
dtype: object
time                   datetime64[ns]
pressure_hpa                  float64
latitude                      float64
longitude                     float64
geopotential_height           float32
temperature                   float32
relative_humidity             float32
u_wind                        float32
v_wind                        float32
dtype: object
(1833005, 9)
(1833005, 9)


In [55]:
print("Old notebook works")

Old notebook works
